In [1]:
import pyiceberg

In [2]:
import trino
import pandas as pd

conn = trino.dbapi.connect(
    host="trino-coordinator",
    port=8080,
    user="jovyan",
    http_scheme="http",
)

cur = conn.cursor()
cur.execute("SELECT version()")
cur.fetchall()

[['476']]

## NESSIE CATALOG

In [3]:
cur.execute("SHOW CATALOGS")
catalogs = cur.fetchall()
catalogs

[['iceberg'], ['rest'], ['system']]

In [4]:
CATALOG = "iceberg"  # поменяй на свой catalog из SHOW CATALOGS

cur.execute(f"SHOW SCHEMAS FROM {CATALOG}")
cur.fetchall()

[['information_schema'], ['system']]

In [5]:
TABLE = "events"
SCHEMA = "my"

cur.execute(f"""
CREATE SCHEMA {CATALOG}.{SCHEMA}
""")

cur.execute(f"""

CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{TABLE} (
    event_id BIGINT,
    user_id VARCHAR,
    event_type VARCHAR,
    amount DOUBLE,
    created_at TIMESTAMP(6)
)
WITH (
    format = 'PARQUET'
)
""")

In [6]:
cur.execute(f"""
INSERT INTO {CATALOG}.{SCHEMA}.{TABLE}
VALUES
    (1, 'user-001', 'click', 10.5, TIMESTAMP '2026-05-12 16:10:00'),
    (2, 'user-002', 'view', 0.0, TIMESTAMP '2026-05-12 16:11:00'),
    (3, 'user-001', 'purchase', 99.9, TIMESTAMP '2026-05-12 16:12:00')
""")

In [7]:
cur.execute(f"""
SELECT *
FROM {CATALOG}.{SCHEMA}.{TABLE}
ORDER BY event_id
""")

rows = cur.fetchall()
rows

[[1, 'user-001', 'click', 10.5, datetime.datetime(2026, 5, 12, 16, 10)],
 [2, 'user-002', 'view', 0.0, datetime.datetime(2026, 5, 12, 16, 11)],
 [3, 'user-001', 'purchase', 99.9, datetime.datetime(2026, 5, 12, 16, 12)]]

In [8]:
df = pd.read_sql(
    f"""
    SELECT *
    FROM {CATALOG}.{SCHEMA}.{TABLE}
    ORDER BY event_id
    """,
    conn,
)

df

/tmp/ipykernel_114/2867425955.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,event_id,user_id,event_type,amount,created_at
0,1,user-001,click,10.5,2026-05-12 16:10:00
1,2,user-002,view,0.0,2026-05-12 16:11:00
2,3,user-001,purchase,99.9,2026-05-12 16:12:00


In [9]:
# Polaris REST Catalog

In [4]:
CATALOG = "rest"  # поменяй на свой catalog из SHOW CATALOGS

cur.execute(f"SHOW SCHEMAS FROM {CATALOG}")
cur.fetchall()

[['information_schema'], ['system']]

In [5]:
TABLE = "events"
SCHEMA = "my"

cur.execute(f"""
CREATE SCHEMA {CATALOG}.{SCHEMA}
""")

cur.execute(f"""

CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{TABLE} (
    event_id BIGINT,
    user_id VARCHAR,
    event_type VARCHAR,
    amount DOUBLE,
    created_at TIMESTAMP(6)
)
WITH (
    format = 'PARQUET'
)
""")